Land-use change classification of a pixel by applying different rules 

Per https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/6837cb74-6c60-800a-b108-fc89b2f766b8

In [60]:
'''
Qs: 
Unstocked Forest Qs:
Tall veg to cultivated short veg is considered immediate forest to grassland conversion. 
Otherwise, tall veg to non-cultivated short veg only counts as grassland conversion if the number of years of consecutive grassland years exceeds n_years_unstocked_forest. 
If something starts out as short veg (non-cultivated) should we consider that Forest or Grassland depending on other circumstances? 
- Check short veg with open woodlands in Africa to see if areas that should be forest are not being classified correctly? 

Is the logging concessions data rasterized? What are the raster values (i.e. 0/1)? Should we add in dataset-specific year logic or just assume logging/ "Forest Land" the entire time period? 
- Canada: 2016; 
- Equatorial Guinea: 2013; 
- Indonesia: 2021; 
- Liberia: 2016; 
- Malaysia: 2010; 
- Republic of the Congo: 2013
- Otherwise: unknown

What gets priority, our unstocked forest/ "Forest remaining Forest" exceptions or cultivated grasslands? Should the exceptions only be where short veg is not cultivated? 
- i.e If number of terminal grassland years < n_years_unstocked_forest but it is cultivated grass 
- i.e. If last year is cultivated grassland, but driver == Logging

Shifting Cultivation Qs:
Proposed rules said to classify as "Cropland" if mix of short veg/ tall veg and driver == Shifting cultivation. 
- Instead, if it starts out as forest, assume forest until short veg or cropland occurs. Once short veg or cropland occurs, assume cropland for the rest of the timeseries if driver == Shifting cultivation. 
- Otherwise, if it starts out as short veg, assume cropland for the whole time series if driver ==  Shifting cultivation. 
Proposed rule said to classify as "Cropland" + driver == Shifting cultivation. Do we want to extend "Cropland" classification in the timeseries when driver != Shifting cultivation?
Should we use n_years_fallow_cropland to determine transition from cropland back to forest? 


Cautions: 
- SDPT planted forests assumed to be Forest remaning Forest only after forests first appear. Establishment year is not currently being used!
- Logging concessions are assumed to be Forest remaning Forest for the entire timeseries.
- No shifting cultivation establishment year. So assuming Cropland the first time tall --> short veg or after "Cropland" LC class.



TODOs: 
How to handle LC -> LU transitions within 5-year intervals if n_years < 5? 
Cultivated short veg == global pasture watch + short veg? Or something else? 

'''

import re
import pandas as pd
import numpy as np

In [61]:
n_years_unstocked_forest = 3    # number of years forest is allowed to be unstocked before being considered a forest --> grassland conversion
#n_years_fallow_cropland = 5    # number of years cropland is allowed to be fallow (i.e. shifting cultivation) before being considered a cropland --> forest conversion

years = [2000, 2005, 2010, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

lc_code_map = {
    0: "bare",
    1: "short veg dry",
    2: "cultivated dry",
    3: "forest dry",
    4: "wetland",
    5: "short veg wet",
    6: "cultivated wet",
    7: "forest wet",
    8: "water",
    9: "snow/ice",
    10: "cropland",
    11: "built up",
}

driver_code_map = {
    1: "Permanent agriculture",
    2: "Hard commodities",
    3: "Shifting cultivation",
    4: "Logging",
    5: "Wildfire",
    6: "Settlements & infrastructure",
    7: "Other natural disturbances"
}

sdpt_code_map = {
    1: "oil palm",
    2: "wood fiber",
    3: "other",
}

In [62]:
# Default LC -> LU regex patterns (numeric codes as strings)
forest_lc = {3, 7}              # Tall vegetation
non_cult_grass_lc = {1, 5}      # Non-cultivated short vegetation
cult_grass_lc = {2, 6}          # Cultivated short vegetation
grass_lc = non_cult_grass_lc | cult_grass_lc
cropland_lc = {10}              # Cropland
settlement_lc = {11}            # Built up
wetland_lc = {4}                # Wetland
other_lc = {0, 8, 9}            # Bare, water, snow/ice

def token_for_lc(v):
    if v in forest_lc:          return "F"
    if v in non_cult_grass_lc:  return "N"
    if v in cult_grass_lc:      return "G"
    if v in cropland_lc:        return "C"
    if v in settlement_lc:      return "S"
    if v in wetland_lc:         return "W"   
    if v in other_lc:           return "O"   
    return "-"

In [63]:
# Configuration
file_path = "/mnt/c/GIS/git/AFOLU_GHG_flux_model/src/LULUCF/scripts/postprocessing/conversion_LUC_scenarios.xlsx"  # Update if needed
sheet_name = "scenarios"

lc_cols = [f"lc_{y}" for y in years]

In [64]:
# Classification logic using regex
# def classify_by_regex(scenario_id, ts_list, regex_rules, driver=None):
#    
#     driver_str = str(driver).strip().lower() if pd.notna(driver) else ""
#     driver_code = driver_code_map.get(driver_str, "")
#         
#     ts_str = '-'.join(map(str, ts_list))   # Creates the string of LC codes
# 
#     if driver_code:
#         ts_str += f"-{driver_code}"
#     
#     print(f"Scenario {scenario_id}: {ts_str}")
#     for i, (pattern, shifting_ag, LUC, conversion) in enumerate (regex_rules, start=1):
#         if re.search(pattern, ts_str):
#             return shifting_ag, LUC, conversion, i
#     return False, False, False, None

def classify_by_regex(scenario_id, ts_list, regex_rules, driver=None, sdpt=None, logging_concession=None):
    # Normalize land cover to ints
    lc_vals = [int(v) for v in ts_list]

    # Initialize empty array for land use timeseries
    lu_vals = []
    for v in lc_vals:
        s = str(v)
        assigned = None
        for rule in regex_rules:
            if rule["kind"] == "default" and rule["pattern"].match(s):
                assigned = rule["label"]; break
        lu_vals.append(assigned)

    # Build tokens/sequence once for exception rules
    tokens    = [token_for_lc(v) for v in lc_vals]
    token_seq = "".join(tokens)

    # Override defaults if exception occured
    ctx = {
        "scenario_id": scenario_id,
        "lc_vals": lc_vals,
        "lu_vals": lu_vals,
        "tokens": tokens,
        "token_seq": token_seq,
        "driver": driver,
        "sdpt": sdpt,
        "logging_concession": logging_concession or 0,
        "n_unstocked": n_years_unstocked_forest
    }
    
    for rule in regex_rules:
        if rule["kind"] == "exception":
            rule["apply"](ctx)

    return lu_vals

In [65]:
# Iterate through scenarios and classify
# def classify_dataframe_regex(df, regex_rules):
#     results = []
#     for idx, row in df.iterrows():
#         driver = row.iloc[1]  # Driver
#         ts_list = row.iloc[2:].tolist()  # Skip the first column (scenario ID)
#         scenario_id = row.iloc[0]        # Use the actual scenario ID
#         shifting_ag, luc_class, conversion_occurred, rule_number = classify_by_regex(scenario_id, ts_list, regex_rules, driver=driver)
#         results.append({
#             "shifting agriculture": shifting_ag,
#             "LUC_class": luc_class,
#             "conversion_occurred": conversion_occurred,
#             "rule_number": rule_number
#         })
#     return pd.DataFrame(results)



def classify_dataframe_regex(df, regex_rules):
    out = []
    for idx, row in df.iterrows():
        driver = int(row["driver"]) if pd.notna(row["driver"]) else None
        sdpt   = int(row["sdpt"])   if pd.notna(row["sdpt"])   else None
        logc   = int(row["logging_concession"]) if pd.notna(row["logging_concession"]) else None
        ts     = [row.get(c) for c in lc_cols]

        lu = classify_by_regex(row["id"], ts, regex_rules, driver=driver, sdpt=sdpt, logging_concession=logc)

        # Create annual land use columns
        lu_cols = {f"lu_{y}": lu[i] for i,y in enumerate(years)}

        # Create time step transitions and conversion flag
        conversion = False
        trans_cols = {}
        for i,(a,b) in enumerate(zip(years[:-1], years[1:])):
            a_lu, b_lu = lu[i], lu[i+1]
            key = f"{a}_{b}"
            if a_lu is None or b_lu is None:
                trans_cols[key] = None
            elif a_lu == b_lu:
                trans_cols[key] = f"{a_lu} remaining {b_lu}"
            else:
                trans_cols[key] = f"{a_lu} to {b_lu}"
                conversion = True

        out.append({
            "id": row["id"],
            "driver": driver_code_map.get(driver, str(driver) if driver is not None else None),
            "sdpt":   sdpt_code_map.get(sdpt,   str(sdpt)   if sdpt   is not None else None),
            "logging_concession": "yes" if logc == 1 else "no",
            **lu_cols,
            **trans_cols,
            "conversion_occurred": conversion
        })
    return pd.DataFrame(out)

In [66]:
# Load data
scenarios_df = pd.read_excel(file_path, sheet_name=sheet_name)

In [67]:
# Define regex-based LUC/conversion rules
#TODO: delete unneccessary keys name, kind
regex_rules = [
    # ---- Default assignments ----
    {"name":"forest_default",      "kind":"default",  "pattern": re.compile(r"^(7|3)$"),        "label":"Forest land"},
    {"name":"grass_default",       "kind":"default",  "pattern": re.compile(r"^(6|5|2|1)$"),    "label":"Grassland"},
    {"name":"wetlands_default",    "kind":"default",  "pattern": re.compile(r"^(4)$"),          "label":"Wetlands"},
    {"name":"cropland_default",    "kind":"default",  "pattern": re.compile(r"^(10)$"),         "label":"Cropland"},
    {"name":"settlements_default", "kind":"default",  "pattern": re.compile(r"^(11)$"),         "label":"Settlements"},
    {"name":"other_default",       "kind":"default",  "pattern": re.compile(r"^(9|8|0)$"),      "label":"Other land"},
    
    # ---- Exceptions; override defaults ----
    # If logging concession, assume all years of short veg lc are unstocked forest so land use remains "Forest Land" 
    {"name":"forest_logging_concessions", "kind":"exception",
        "apply": lambda ctx: (ctx["lu_vals"].__setitem__(slice(None), ["Forest land" if v is not None else None for v in ctx["lc_vals"]])
                              if ctx["logging_concession"] == 1 else None)
    },
    
    # If SDPT is planted forest, assume any short veg lc after forest lc occurs is unstocked forest so land use remains "Forest Land"
    # Note: Using first time forest lc occurs as a proxy for planted forest "establishment year"
    {"name":"forest_sdpt", "kind":"exception",
        "apply": lambda ctx: (
         (lambda start_idx: [
             ctx["lu_vals"].__setitem__(i, "Forest land")
             for i in range(start_idx, len(ctx["lc_vals"]))
             if ctx["lc_vals"][i] in grass_lc
         ])(
             (lambda: next(
                 (j for j in range(1, len(ctx["lc_vals"]))
                  if ctx["lc_vals"][j-1] in forest_lc and ctx["lc_vals"][j] in grass_lc),
                 None
             ))()
         )
     ) if (ctx["sdpt"] == 2 and re.search(r"F[NG]", ctx["token_seq"])) else None
    },
    
    # If tall veg switches to non-cultivated short veg for less than n_years_unstocked_forest and returns back to tall veg, assume unstocked forest so land use remains "Forest Land" 
    # If tall veg switches to cultivated short veg then keep "Forest to Grassland" conversion
    {"name":"forest_temp_unstocked_forest", "kind":"exception",
         "apply": lambda ctx: [
         (lambda a,b: [ctx["lu_vals"].__setitem__(k, "Forest land") for k in range(a, b)])(
             *re.Match.span(m, 1)  # start/end of the N-run
         )
         for m in re.finditer(rf"F(N{{1,{ctx['n_unstocked']}}})F", ctx["token_seq"])
     ]
    },
    
    # If timeseries end with short veg after tall veg, assume unstocked forest if driver is "Logging" so land use remains "Forest Land" 
    {"name":"forest_logging", "kind":"exception",
        "apply": lambda ctx: (
         (lambda: [
             (lambda: None)()
         ])()
     ) if ctx["driver"] == 4 else None
    },
    
    # If tall veg switches to non-cultivated short veg at the end of the timeseries for less than n_years_unstocked_forest, assume unstocked forest so land use remains "Forest Land" 
    {"name":"forest_temp_unstocked_terminal", "kind":"exception",
         "apply": lambda ctx: (
         (lambda m: [
             ctx["lu_vals"].__setitem__(i, "Forest land")
             for i in range(len(ctx["lc_vals"]) - len(m.group(1)), len(ctx["lc_vals"]))
         ])(re.match(r".*?(N+)$", ctx["token_seq"]))
     ) if ((m := re.match(r".*?(N+)$", ctx["token_seq"])) and 0 < len(m.group(1)) <= ctx["n_unstocked"]) else None
    },

    
    # If driver is shifting cultivation: 
    # If series starts as tall veg but switches to short veg or cropland at some point, any subsequent tall veg is assumed to be fallow cropland so land use is considered "Cropland" 
    # Otherwise,all tall veg in the timeseries is assumed to be fallow cropland so land use is considered "Cropland" 
    {"name":"cropland_shifting_cultivation", "kind":"exception",
     "apply": lambda ctx: (
         (lambda startF: (
             (lambda first_sw: first_sw is not None and [
                 ctx["lu_vals"].__setitem__(i, "Cropland")
                 for i in range(first_sw, len(ctx["lc_vals"])) if ctx["tokens"][i] == "F"
             ])(next((i for i,tok in enumerate(ctx["tokens"]) if tok in {"N","G","C"}), None))
             if startF and any(tok in {"N","G","C"} for tok in ctx["tokens"]) else
             [ctx["lu_vals"].__setitem__(i, "Cropland") for i,tok in enumerate(ctx["tokens"]) if tok == "F"]
         ))(len(ctx["tokens"])>0 and ctx["tokens"][0] == "F")
     ) if ctx["driver"] == 3 else None
    },
]

In [68]:
# Run classification
#results_df = classify_dataframe_regex(scenarios_df, regex_rules)

# Coerce scenarios_df to numeric
for c in lc_cols + ["driver", "sdpt", "logging_concession"]:
    if c in scenarios_df.columns: scenarios_df[c] = pd.to_numeric(scenarios_df[c], errors="coerce")
    
# Run classification
results_df = classify_dataframe_regex(scenarios_df, regex_rules)

# Output results 
print("\nClassification Results:")
print(results_df)


Classification Results:
      id                driver        sdpt logging_concession      lu_2000  \
0    1.0                  None        None                 no  Forest land   
1    2.0               Logging        None                 no  Forest land   
2    3.0                  None  wood fiber                 no  Forest land   
3    4.0                  None       other                 no  Forest land   
4    5.0                  None        None                yes  Forest land   
5    6.0                  None        None                 no  Forest land   
6    7.0                  None        None                 no    Grassland   
7    8.0                  None        None                 no     Cropland   
8    9.0                  None        None                 no  Forest land   
9   10.0  Shifting cultivation        None                 no  Forest land   
10  11.0  Shifting cultivation        None                 no  Forest land   
11  12.0  Shifting cultivation        N

In [69]:
results_df.to_excel("/mnt/c/GIS/git/AFOLU_GHG_flux_model/src/LULUCF/scripts/postprocessing/conversion_LUC_scenario_results.xlsx", index=False)